# <font color='blue'> AgentsVille Trip Planner </font>


#### <font color='red'> Content: Building Your AI Travel Assistant </font>

Your "AgentsVille Trip Planner" will be a Jupyter Notebook application that orchestrates interactions with a Large Language Model (LLM) to perform two main functions:

1. **The Expert Planner (Initial Itinerary Generation)**:
   
    - **Your Task**: Based on a set of user-defined travel preferences (destination, duration, interests, budget), you will prompt the AI to act as an expert travel planner.
    - **The Outcome**: The AI must generate a detailed, day-by-day travel itinerary for AgentsVille. This itinerary needs to be a structured JSON object that conforms to a predefined Pydantic model. This stage will heavily rely on your ability to craft prompts that elicit complex, structured output, guiding the LLM through a planning process.



2. **The Resourceful Assistant (Itinerary Enhancement with a Tool-Using ReAct Agent)**:
   
    - **Your Task**: Once an initial itinerary is generated, the user might have follow-up questions or request modifications. Additionally, we may have other requirements to satisfy before finalizing the itinerary. You will design an "AI Travel Assistant".
    - **The Outcome**: This agent will analyze the user's request in the context of the current itinerary. It will then decide if any of its available "tools" — like calling the simulated activities API tool — can assist.
        - If a tool is needed, the agent will THINK about its plan and ACT by generating a structured request for the tool.
        - Your Python code will simulate the tool's execution and provide the result back to the agent as an OBSERVATION.
        - The agent will then use this observation to continue its reasoning or formulate a final, helpful answer to the user.


### Outline

    1. Define Vacation Details
    2. Review Weather and Activity schedules
    3. The ItineraryAgent
    4. Evaluating the Itinerary
    5. Defining the Tools
    6. The ItineraryRevisionAgent



In [4]:
from openai import OpenAI
from typing import Optional
from IPython.display import Markdown, display
from pprint import pprint

import os
import pandas as pd

import sys
import json
import re
import datetime

sys.path.append("/Users/hhung/Desktop/udacity_agentic_ai/")

with open('/Users/hhung/Desktop/udacity_agentic_ai/api_key.json', 'r') as file:
    api_keys = json.load(file)
    openai_key = api_keys["openai"]

# print(openai_key)
client = OpenAI(api_key=openai_key)

from utils import (
    display_responses,
    get_completion,
    print_in_box,
    MODEL,
    OpenAIModels,
)
from utils_travel_agent import (
    ChatAgent,
    VacationInfo,
    Traveler,
    TravelPlan,
    call_activities_api_mocked,
    call_weather_api_mocked,
    do_chat_completion,
)

print(f"LLM model: {MODEL}")

LLM model: gpt-4.1-mini


In [5]:
# Install required packages if not already installed
# No changes needed here.
%pip install -q json-repair==0.47.1 numexpr==2.11.0 openai==1.74.0 pandas==2.3.0 pydantic==2.11.7 python-dotenv==1.1.0

Note: you may need to restart the kernel to use updated packages.


## 1. Define Vacation Details

### Instructions

* Specify the trip duration, interests, and constraints.
* Use Pydantic to structure and verify this information in a class called `VacationInfo`.

In [6]:
# The Vacation Info Data Structure

VACATION_INFO_DICT = {
    "travelers": [
        {
            "name": "Yuri",
            "age": 30,
            "interests": ["tennis", "cooking", "comedy", "technology"],
        },
        {
            "name": "Hiro",
            "age": 25,
            "interests": ["reading", "music", "theatre", "art"],
        },
    ],
    "destination": "AgentsVille",
    "date_of_arrival": "2025-06-10",  # Mock data exists for 2025-06-10
    "date_of_departure": "2025-06-12",  # ...until 2025-06-15.
    "budget": 130,  # Budget is in fictional currency units.
}

In [7]:
# Validate the VacationInfo data structure
vacation_info = VacationInfo.model_validate(VACATION_INFO_DICT)

# Display the vacation info as a dictionary
pprint(vacation_info.model_dump())

# Check that VacationInfo contains the expected data
assert "travelers" in vacation_info.model_dump().keys(), "VacationInfo should contain 'travelers' key"
assert "destination" in vacation_info.model_dump().keys(), "VacationInfo should contain 'destination' key"
assert "date_of_arrival" in vacation_info.model_dump().keys(), "VacationInfo should contain 'date_of_arrival' key"
assert "date_of_departure" in vacation_info.model_dump().keys(), "VacationInfo should contain 'date_of_departure' key"
assert "budget" in vacation_info.model_dump().keys(), "VacationInfo should contain 'budget' key"
assert isinstance(vacation_info.travelers, list), "Travelers should be a list"
assert all(isinstance(traveler, Traveler) for traveler in vacation_info.travelers), "All travelers should be instances of Traveler class"
assert isinstance(vacation_info.date_of_arrival, datetime.date), "date_of_arrival should be a date"
assert isinstance(vacation_info.date_of_departure, datetime.date), "date_of_departure should be a date"
assert isinstance(vacation_info.budget, int), "budget should be an integer"

# If all assertions pass, print a success message
print("✅ VacationInfo data structure is valid!")

{'budget': 130,
 'date_of_arrival': datetime.date(2025, 6, 10),
 'date_of_departure': datetime.date(2025, 6, 12),
 'destination': 'AgentsVille',
 'travelers': [{'age': 30,
                'interests': [tennis, cooking, comedy, technology],
                'name': 'Yuri'},
               {'age': 25,
                'interests': [reading, music, theatre, art],
                'name': 'Hiro'}]}
✅ VacationInfo data structure is valid!


## 2. Review Weather and Activity schedules

We will call an API to get all the data **at once**, in order to be able to include it in the context for our itinerary planning agent.

Also, we will format the data as Pandas DataFrames for easier viewing. Take the time to read and understand the data to see if the agent notices the same things you do. For instance:
- What does the weather look like for the trip? On what days it is sunny, rainy, or cloudy?
- What activities are available on each day? Are there any special events or festivals related to the user's interests?

### Instructions

* Simulate API calls to gather weather data and available activities in bulk
* Review the data manually to understand the available options


### 2.1 Activity Data

In [8]:

pd.set_option("display.max_colwidth", None)  # Show full content in DataFrame cells

weather_for_dates = [
    call_weather_api_mocked(
        date=ts.strftime("%Y-%m-%d"), city=vacation_info.destination
    )
    for ts in pd.date_range(
        start=vacation_info.date_of_arrival, 
        end=vacation_info.date_of_departure,
        freq="D",
    )
]

weather_for_dates_df = pd.DataFrame(weather_for_dates)

weather_for_dates_df

,date,city,temperature,temperature_unit,condition,description
0,2025-06-10,AgentsVille,31,celsius,clear,A bright and sunny day in AgentsVille with clear skies and warm temperatures. Perfect weather for outdoor activities!
1,2025-06-11,AgentsVille,34,celsius,partly cloudy,"A warm day with periods of sunshine and mixed clouds, making it a perfect opportunity for outdoor activities."
2,2025-06-12,AgentsVille,28,celsius,thunderstorm,"A thunderstorm is expected to roll in during the afternoon, bringing heavy rain and gusty winds. The atmosphere will feel charged with humidity, creating a sultry and dramatic setting as clouds build in the sky."


### 2.2 Activity Data

In [9]:

activities_for_dates = [
    activity
    for ts in pd.date_range(
        start=vacation_info.date_of_arrival, 
        end=vacation_info.date_of_departure,
        freq="D",
    )
    for activity in call_activities_api_mocked(
        date=ts.strftime("%Y-%m-%d"), city=vacation_info.destination
    )
]

activities_for_dates_df = pd.DataFrame(activities_for_dates)

activities_for_dates_df

,activity_id,name,start_time,end_time,location,description,price,related_interests
0,event-2025-06-10-0,FutureTech Breakfast Meet-Up,2025-06-10 09:00,2025-06-10 11:00,"The Innovation Atrium, Tech District, AgentsVille","Join fellow technology enthusiasts for a dynamic morning at the FutureTech Breakfast Meet-Up! Dive into the latest trends in tech, gadget demos, and networking opportunities over coffee and fresh pastries. Held indoors at the spacious Innovation Atrium, this event is perfect for tech lovers eager to exchange ideas and discover new possibilities in a comfortable, modern setting.",20,[technology]
1,event-2025-06-10-1,Serve & Savor: Tennis and Taste Luncheon,2025-06-10 12:00,2025-06-10 13:30,"The Grand Racquet Terrace, AgentsVille","Join us for 'Serve & Savor,' the ultimate crossover event for cooking and tennis enthusiasts in AgentsVille! Kick off your lunch hour with a friendly round of doubles on our outdoor courts, then unwind with a hands-on cooking workshop led by a local chef, where you'll prepare and enjoy delicious energy-boosting recipes. Whether you come for the sport or the flavors, this energizing luncheon celebrates both passions in a lively outdoor setting. Ideal for anyone who loves to play, cook, or simply savor fresh food and fun!",20,"[cooking, tennis]"
2,event-2025-06-10-2,Artful Athletics: Paint & Play Extravaganza,2025-06-10 15:00,2025-06-10 17:00,"Creative Courts Park, AgentsVille","Join us for an exciting afternoon at Creative Courts Park, where the worlds of art and sports collide! At 'Artful Athletics: Paint & Play Extravaganza', you'll participate in collaborative outdoor murals inspired by your favorite sports, and then get moving with fun, friendly sports mini-games. Whether you love painting or playing, this event celebrates creativity, teamwork, and the joy of movement under the open sky. Perfect for art lovers and sports enthusiasts alike—come ready to express yourself and get active! (Event is held outdoors; in case of rain, we move indoors to the Community Gym nearby.)",15,"[art, sports]"
3,event-2025-06-10-3,AgentsVille Twilight Writing Escape,2025-06-10 19:00,2025-06-10 21:00,"The Ink Loft, 12 Quill Lane, AgentsVille","Join fellow writers for an inspiring evening at The Ink Loft, where words flow as freely as the coffee! This writing-themed event welcomes all—novelists, poets, bloggers, or anyone with a passion for storytelling. Set indoors in AgentsVille's coziest lounge, enjoy writing games, group prompts, and opportunities to read your work aloud. Connect, create, and celebrate the art of writing in this creative indoor haven.",15,"[writing, reading, art]"
4,event-2025-06-11-0,Morning Groove Dance Party,2025-06-11 09:00,2025-06-11 10:30,"Rhythm Hall, Center Plaza, AgentsVille","Start your day with energy and joy at the Morning Groove Dance Party! This lively event welcomes dancers of all levels to join a vibrant indoor session filled with upbeat music and fun routines. Whether you love modern pop, Latin beats, or classic disco, our dance instructors will guide you to move and groove. Connect with fellow dance lovers in the colorful atmosphere of Rhythm Hall. Perfect for fans of dancing, music, and fitness. Let the rhythm move you! (Indoor event.)",15,"[dancing, music, fitness]"
5,event-2025-06-11-1,Tech Lunch & Learn: AI Frontiers,2025-06-11 12:00,2025-06-11 13:30,"The Digital Atrium, AgentsVille","Join fellow tech enthusiasts for a dynamic lunchtime event exploring the future of artificial intelligence! Held indoors at The Digital Atrium, this Tech Lunch & Learn features engaging lightning talks, interactive demos, and networking opportunities all centered around technology and innovation. Enjoy light lunch fare as you connect with others passionate about technology, AI, and the digital world. Whether you're a seasoned developer or just curious about tech, this event is for you! Related interests: technology, music (sound tech demos), photography (AI imaging), writing (AI cr

## 3. The ItineraryAgent

First we will review the Pydantic objects used for defining the output of our agent, the TravelPlan, ItineraryDay, Activity, and Weather classes.

Second, we will create a Chain-of-Thought prompt to guide the agent in planning the trip. This prompt will instruct the agent to consider the weather, activities, and user preferences when creating the itinerary.

Third, we will run the agent to produce the TravelPlan object, which will will refine in the following steps.

### Instructions

* Implement an agent that generates a **day-by-day** itinerary based on the vacation details
* The system prompt will guide the agent's reasoning through a step-by-step planning process to take travel preferences (e.g. destination, dates, interests) and generate a detailed day-by-day itinerary
* Craft the components of the prompt (including the role, task/instructions, output format, examples, and context) to elicit the best possible itinerary in one LLM cal


In [10]:
# for ItineraryAgent
ITINERARY_AGENT_SYSTEM_PROMPT = f"""
You are a professional travel plan assistant, to help make a day-by-day travel plan for our customer, based on vacation details: date ranges, destinations, weather conditions on the days, and the budget.

Think step by step.

## Task

The day-by-day travel plan should satisfy the following conditions:
- One activitiy per day.
- The activities should follow travelers' interests.
- The activities should be suitable for the weather conditions. For example, if the weather forecast shows a storm, we should try not to schedule outdoor activities.
- Each activity only appears once during the entire trip.
- Should avoid to make a plan which has outdoor activities only during the itinerary. 
- For each day, the activity duration should not be longer than 10 hours.
- The travel plan needs to estimates the total_cost, given by sum over all activities's prices during the entire trip.


## Output Format

Respond using two sections (ANALYSIS AND FINAL OUTPUT) in the following format:

    ANALYSIS:
    [structured analysis]
    
    - First list the required conditions from the vacation info.
    - Then Summarize your plan.

    TRAVEL PLANS:
    ```json
    {json.dumps(TravelPlan.model_json_schema(), indent=2)}
    ```

## Context

Weather Data:
{json.dumps(weather_for_dates, indent=2)}

Activity Data:
{json.dumps(activities_for_dates, indent=2)}

Travel's vacation info:
{vacation_info.model_dump_json(indent=2)}
"""


In [11]:
assert "TASK" in ITINERARY_AGENT_SYSTEM_PROMPT.upper(), "❌ ITINERARY_AGENT_SYSTEM_PROMPT should contain a 'TASK' section"
assert "OUTPUT FORMAT" in ITINERARY_AGENT_SYSTEM_PROMPT.upper(), "❌ ITINERARY_AGENT_SYSTEM_PROMPT should contain a 'OUTPUT FORMAT' section"

In [12]:
type(vacation_info), vacation_info.model_dump_json

(utils_travel_agent.VacationInfo,
 <bound method BaseModel.model_dump_json of VacationInfo(travelers=[Traveler(name='Yuri', age=30, interests=[tennis, cooking, comedy, technology]), Traveler(name='Hiro', age=25, interests=[reading, music, theatre, art])], destination='AgentsVille', date_of_arrival=datetime.date(2025, 6, 10), date_of_departure=datetime.date(2025, 6, 12), budget=130)>)

In [13]:
def get_itinerary(client, vacation_info: VacationInfo, model: Optional[OpenAIModels] = None) -> TravelPlan:
    """Generates a travel itinerary based on the provided vacation information."""
    response = get_completion(client, 
                              system_prompt=ITINERARY_AGENT_SYSTEM_PROMPT, 
                              user_prompt=vacation_info.model_dump_json(indent=2),
                              model=model)

    print_in_box(response, "Raw Response")

    # Parse the response
    json_text = response.strip()

    if "```json" in json_text:
        json_text = json_text.split("```json")[1].split("```")[0].strip()
    
    try:
        travel_plan = TravelPlan.model_validate_json(json_text)
        return travel_plan
    except Exception as e:
        print("Error validating the following text as TravelPlan JSON:")
        print(json_text)
        raise

travel_plan_1 = get_itinerary(
    client,
    vacation_info=vacation_info,
    model=MODEL,  # Optionally, you can change the model here
)

if travel_plan_1 is not None:
    print("✅ Initial itinerary generated successfully. Congratulations!")


╔═══════════════════════════════════════════════════[ Raw Response ]═══════════════════════════════════════════════════╗
║ ANALYSIS:                                                                                                            ║
║ - Required conditions from vacation info:                                                                            ║
║   * Destination: AgentsVille                                                                                         ║
║   * Travel dates: June 10 to June 12, 2025 (3 days)                                                                  ║
║   * Budget: 130                                                                                                      ║
║   * Interests combined from Yuri and Hiro: tennis, cooking, comedy, technology, reading, music, theatre, art         ║
║ - Weather conditions:                                                                                                ║
║   * June 10: Clear, warm, sui

In [14]:
travel_plan_1.total_cost

60

In [15]:
travel_plan_1.itinerary_days

[ItineraryDay(date=datetime.date(2025, 6, 10), weather=Weather(temperature=31.0, temperature_unit='celsius', condition='clear'), activity_recommendations=[ActivityRecommendation(activity=Activity(activity_id='event-2025-06-10-1', name='Serve & Savor: Tennis and Taste Luncheon', start_time=datetime.datetime(2025, 6, 10, 12, 0), end_time=datetime.datetime(2025, 6, 10, 13, 30), location='The Grand Racquet Terrace, AgentsVille', description="Join us for 'Serve & Savor,' the ultimate crossover event for cooking and tennis enthusiasts in AgentsVille! Kick off your lunch hour with a friendly round of doubles on our outdoor courts, then unwind with a hands-on cooking workshop led by a local chef, where you'll prepare and enjoy delicious energy-boosting recipes. Whether you come for the sport or the flavors, this energizing luncheon celebrates both passions in a lively outdoor setting. Ideal for anyone who loves to play, cook, or simply savor fresh food and fun!", price=20, related_interests=[c

## 4. Evaluating the Itinerary

We've successfully created an itinerary, but how do we know if it's any good?

Now we will create some evaluation functions (sometimes called evals) to help us determine the quality of the itinerary. We will not only want our final output to be of the highest quality possible initially, but we also want to give the chance for the LLM to reflect on its own output and make improvements at a second pass.

If the itinerary does not meet all the criteria specified here, no worries! Afterwards, we will implement a feedback loop that will allow the agent to improve its plan iteratively.

### Instructions

Evaluate the itinerary using a set of criteria to ensure a high-quality travel plan:
- For instance, does the itinerary match the city and the dates requested?
- Or, is the total cost calulation accurate and is it within budget?
- Or, does the agent hallucinate any activities that are not available?
- Or, does the agent suggest activities that are not suitable for the weather? This specific evaluation function will require the use of an - LLM to compare the event description against the weather data.

In [16]:
from eval_travel_agent import (
    get_eval_results,
    AgentError,
)

In [17]:
# Basic evaluation functions

from eval_travel_agent import eval_start_end_dates_match

get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[eval_start_end_dates_match],
)

EvaluationResults(success=True, failures=[], eval_functions=['eval_start_end_dates_match'])

In [18]:
# Evaluation functions related to the budget and total cost

from eval_travel_agent import (
    eval_total_cost_is_accurate,
    eval_total_cost_is_within_budget,
)

get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[eval_total_cost_is_accurate, eval_total_cost_is_within_budget],
)

EvaluationResults(success=True, failures=[], eval_functions=['eval_total_cost_is_accurate', 'eval_total_cost_is_within_budget'])

In [19]:
# The final output contains copies of the activities, so we need to verify that the copies are accurate and not hallucinated.

from eval_travel_agent import eval_itinerary_events_match_actual_events

get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[eval_itinerary_events_match_actual_events],
)

EvaluationResults(success=True, failures=[], eval_functions=['eval_itinerary_events_match_actual_events'])

In [20]:
# Check that the itinerary includes at least one activity matching each traveler's interests.

from eval_travel_agent import eval_itinerary_satisfies_interests

get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[eval_itinerary_satisfies_interests],
)

✅ Traveler Yuri has a match with interest {cooking, tennis} at Serve & Savor: Tennis and Taste Luncheon
✅ Traveler Yuri has a match with interest {cooking} at Palette & Palate: Art Meets Cooking Experience
✅ Traveler Yuri has a match with interest {technology} at Tech & Film Fusion Night
✅ Traveler Hiro has a match with interest {art} at Palette & Palate: Art Meets Cooking Experience


EvaluationResults(success=True, failures=[], eval_functions=['eval_itinerary_satisfies_interests'])

In [21]:
from eval_travel_agent import eval_activities_and_weather_are_compatible

eval_results = get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=[
        eval_activities_and_weather_are_compatible
    ],
)

eval_results

✅ Activity Serve & Savor: Tennis and Taste Luncheon (on 2025-06-10) and weather 'clear' are compatible.
✅ Activity Palette & Palate: Art Meets Cooking Experience (on 2025-06-11) and weather 'partly cloudy' are compatible.
✅ Activity Tech & Film Fusion Night (on 2025-06-12) and weather 'thunderstorm' are compatible.


EvaluationResults(success=True, failures=[], eval_functions=['eval_activities_and_weather_are_compatible'])

In [22]:
# Run all of the evaluation functions again
# No changes needed here.

ALL_EVAL_FUNCTIONS = [
    eval_start_end_dates_match,
    eval_total_cost_is_accurate,
    eval_itinerary_events_match_actual_events,
    eval_itinerary_satisfies_interests,
    eval_total_cost_is_within_budget,
    eval_activities_and_weather_are_compatible,
]

eval_results = get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=ALL_EVAL_FUNCTIONS,
)

eval_results.model_dump()

✅ Traveler Yuri has a match with interest {cooking, tennis} at Serve & Savor: Tennis and Taste Luncheon
✅ Traveler Yuri has a match with interest {cooking} at Palette & Palate: Art Meets Cooking Experience
✅ Traveler Yuri has a match with interest {technology} at Tech & Film Fusion Night
✅ Traveler Hiro has a match with interest {art} at Palette & Palate: Art Meets Cooking Experience
✅ Activity Serve & Savor: Tennis and Taste Luncheon (on 2025-06-10) and weather 'clear' are compatible.
✅ Activity Palette & Palate: Art Meets Cooking Experience (on 2025-06-11) and weather 'partly cloudy' are compatible.
✅ Activity Tech & Film Fusion Night (on 2025-06-12) and weather 'thunderstorm' are compatible.


{'success': True,
 'failures': [],
 'eval_functions': ['eval_start_end_dates_match',
  'eval_total_cost_is_accurate',
  'eval_itinerary_events_match_actual_events',
  'eval_itinerary_satisfies_interests',
  'eval_total_cost_is_within_budget',
  'eval_activities_and_weather_are_compatible']}

## 5. Defining the Tools

Our ItineraryRevisionAgent will be a ReAct-based agent that will use tools to:
- Evaluate/Re-evaluate the itinerary
- Use a **calculator** since LLMs sometimes struggle with arithmetic
- Call the activities API to get more information about activities
- Return the final itinerary

### Instructions

We will define four tools to assist the agent:
* **calculator_tool**: to accurately calculate costs
* **get_activities_by_date_tool**: to retrieve activities for a specific date
* **run_evals_tool**: to evaluate the itinerary against the criteria
* **final_answer_tool**: to provide the final answer in a structured format

In [23]:
from tools_travel_agent import (
    get_tool_descriptions_string,
    calculator_tool,
    get_activities_by_date_tool,
    run_evals_tool,
    final_answer_tool,
)

###  calculator_tool

In [24]:
# Define the calculator tool that evaluates mathematical expressions.
assert calculator_tool("1 + 1") == 2.0
print(get_tool_descriptions_string([calculator_tool]))

* `calculator_tool`: Evaluates a mathematical expression and returns the result as a float.

    Args:
        input_expression (str): A string containing a valid mathematical expression to evaluate.

    Returns:
        float: The result of the evaluated expression.

    Example:
        >>> calculator_tool("1 + 1")
        2.0
    



### get_activities_by_date_tool

In [25]:
# Tool to fetch activities for a given date and city.
assert len(get_activities_by_date_tool("2025-06-10", "AgentsVille")) > 0
print(get_tool_descriptions_string([get_activities_by_date_tool]))

* `get_activities_by_date_tool`: 
    Retrieve activities given a date and a city names.

    Args:
        date (str): date, yyyy-mm-dd
        city (str): city name

    Returns:
        list[dict]: The list of activity dictionary.

    



### run_evals_tool

In [26]:
# Let's double check that the tool works as expected.
run_evals_tool(vacation_info=vacation_info, travel_plan=travel_plan_1)

✅ Traveler Yuri has a match with interest {cooking, tennis} at Serve & Savor: Tennis and Taste Luncheon
✅ Traveler Yuri has a match with interest {cooking} at Palette & Palate: Art Meets Cooking Experience
✅ Traveler Yuri has a match with interest {technology} at Tech & Film Fusion Night
✅ Traveler Hiro has a match with interest {art} at Palette & Palate: Art Meets Cooking Experience
✅ Activity Serve & Savor: Tennis and Taste Luncheon (on 2025-06-10) and weather 'clear' are compatible.
✅ Activity Palette & Palate: Art Meets Cooking Experience (on 2025-06-11) and weather 'partly cloudy' are compatible.
✅ Activity Tech & Film Fusion Night (on 2025-06-12) and weather 'thunderstorm' are compatible.

╔════════════════════════════════════════════[ ChatAgent - System Prompt ]═════════════════════════════════════════════╗
║ You are an expert in evaluating whether a travel plan incorporates traveler feedback.                                ║
║     ## Output Format                               

{'success': False,
 'failures': ['Traveler feedback was not successfully incorporated into the revised travel plan. Response: NOT_INCORPORATED\nREASON: The revised travel plan provides only one activity per day, which does not fulfill the traveler’s feedback of having at least two activities per day.']}

In [28]:
print(get_tool_descriptions_string([run_evals_tool]))

* `run_evals_tool`: Runs all evaluation tools on the provided travel plan and vacation info.

    Args:
        travel_plan (TravelPlan): The travel plan to evaluate.

    Returns:
        EvaluationResults: The results of the evaluations.
    



### final_answer_tool

In [29]:
print(get_tool_descriptions_string([final_answer_tool]))

* `final_answer_tool`: Returns the final travel plan

    Args:
        final_output (TravelPlan): The final travel plan to return.

    Returns:
        TravelPlan: The final travel plan.
    



### All tools

In [30]:
# List of all tools available for the agent

ALL_TOOLS = [
    calculator_tool,
    get_activities_by_date_tool,
    run_evals_tool,
    final_answer_tool,
]
print(get_tool_descriptions_string(ALL_TOOLS))

* `calculator_tool`: Evaluates a mathematical expression and returns the result as a float.

    Args:
        input_expression (str): A string containing a valid mathematical expression to evaluate.

    Returns:
        float: The result of the evaluated expression.

    Example:
        >>> calculator_tool("1 + 1")
        2.0
    
* `get_activities_by_date_tool`: 
    Retrieve activities given a date and a city names.

    Args:
        date (str): date, yyyy-mm-dd
        city (str): city name

    Returns:
        list[dict]: The list of activity dictionary.

    
* `run_evals_tool`: Runs all evaluation tools on the provided travel plan and vacation info.

    Args:
        travel_plan (TravelPlan): The travel plan to evaluate.

    Returns:
        EvaluationResults: The results of the evaluations.
    
* `final_answer_tool`: Returns the final travel plan

    Args:
        final_output (TravelPlan): The final travel plan to return.

    Returns:
        TravelPlan: The final tr

## 6. The ItineraryRevisionAgent

The ItineraryRevisionAgent will
* take initial feedback from the user about the itinerary and
* use the tools defined above

to refine original itinerary iteratively using a ReAct-based approach.

### Instructions

* We will implement a second agent that revises the itinerary based on feedback using the ReAct `THOUGHT → ACTION → OBSERVATION` cycle.
    - The LLM will first generated a THOUGHT / ACTION message, which contains reasoning steps and a tool call invocation.
    - The Python code will parse the tool call and execute it, returning the result as a string to the LLM in an OBSERVATION message.
    - After this cycle repeats n number of times, the LLM will invoke the `final_answer_tool` to signal to the Python code to end the loop and return the final answer.

* This agent will also **incorporate feedback on the initial itinerary** from the travelers to ensure the final plan has **at least 2 activities per day**. A new evaluation function using a powerful LLM will be created to check this user feedback.

* The agent will use the tools above to refine the plan iteratively, checking the weather and available activities, and ensuring the itinerary meets all constraints.

In [31]:
from eval_travel_agent import eval_traveler_feedback_is_incorporated

ALL_EVAL_FUNCTIONS = [
    eval_start_end_dates_match,
    eval_total_cost_is_accurate,
    eval_itinerary_events_match_actual_events,
    eval_itinerary_satisfies_interests,
    eval_total_cost_is_within_budget,
    eval_activities_and_weather_are_compatible,
    eval_traveler_feedback_is_incorporated,  # Add this new evaluation
]

get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_1,
    eval_functions=ALL_EVAL_FUNCTIONS,
)

✅ Traveler Yuri has a match with interest {cooking, tennis} at Serve & Savor: Tennis and Taste Luncheon
✅ Traveler Yuri has a match with interest {cooking} at Palette & Palate: Art Meets Cooking Experience
✅ Traveler Yuri has a match with interest {technology} at Tech & Film Fusion Night
✅ Traveler Hiro has a match with interest {art} at Palette & Palate: Art Meets Cooking Experience
✅ Activity Serve & Savor: Tennis and Taste Luncheon (on 2025-06-10) and weather 'clear' are compatible.
✅ Activity Palette & Palate: Art Meets Cooking Experience (on 2025-06-11) and weather 'partly cloudy' are compatible.
✅ Activity Tech & Film Fusion Night (on 2025-06-12) and weather 'thunderstorm' are compatible.

╔════════════════════════════════════════════[ ChatAgent - System Prompt ]═════════════════════════════════════════════╗
║ You are an expert in evaluating whether a travel plan incorporates traveler feedback.                                ║
║     ## Output Format                               

EvaluationResults(success=False, failures=['Traveler feedback was not successfully incorporated into the revised travel plan. Response: NOT_INCORPORATED\nREASON: The revised plan does not offer at least two activities per day on any of the days, which does not fulfill the traveler’s feedback.'], eval_functions=['eval_start_end_dates_match', 'eval_total_cost_is_accurate', 'eval_itinerary_events_match_actual_events', 'eval_itinerary_satisfies_interests', 'eval_total_cost_is_within_budget', 'eval_activities_and_weather_are_compatible', 'eval_traveler_feedback_is_incorporated'])

In [32]:
TRAVELER_FEEDBACK = "I want to have at least two activities per day."

In [33]:
ITINERARY_REVISION_AGENT_SYSTEM_PROMPT = f"""

You are a travel planning expert and helping our customers to make a comprehensive travel plan iteratively using tool calls 
and reasoning in a multi-step process.


## Task Instructions:

- You will repeat the following step-by-step reasoning until the itinerary satisfies to travelers' requests and feedbacks:
    - THINKING: Look the day-by-day itinerary, analyze the problems, and determine what tools to call.
    - ACTING: Take a single tool call.
    - OBSERVE: incorporate the response to improve the itinerary.

- The final response must satisfy traveler's feedback {TRAVELER_FEEDBACK}.
- Before providing the final output, you must pass eval tools.
- As soon as you know the final answer, call the tool `final_answer_tool` in an `ACT` message and stop iteration.
- ALWAYS provide a tool call in ACT; otherwise you will fail.

- You will always respond with a single THOUGHT/ACTION message of the following format:
    - THOUGHT:
        * First, you will reason about the problems and determine what the next logical action needs to take.
        * Second, incorporate the previous response, and determine action steps:
            (1) For each day in the itinerary, check the number of activities. If a certain day has only one activitiy, you must call the tool to add at least one more activity to that day. This is crucial for ensuring a well-rounded itinerary.
            (2) If the total_cost exceeds the budget, need to replace a similar cheaper-price activity.
            (3) If certain activities are not suitable to the weather conditions, replace a similar indoor activity.
            (4) If there exist duplicated activities, replace a similar but different activity.

    - ACTION:
        * Based on your thought process, you will call ONE of the available tools.
          Available tools are [`get_activities_by_date_tool`, `calculator_tool`, `run_evals_tool`, `final_answer_tool`].

            - If response shows success, `call final_answer_tool(final_output={{....}})`
            - If response shows failure and need to retreieve activity data, call `get_activities_by_date_tool(date=.., city=..)`:
                * If there is a not-suitbale activity due to weather or not match interest
                * If need to add an activity to make sure at least two activities per day
        * Calling tool should be in format using json {{"tool_name": "[tool_name]", "arguments": {{"arg1": "value1", ...}}}}



## Available Tools

* `calculator_tool(input_expression: str)`: Calculate the estimated cost for the entire trip
    - Example:
        - Input: `ACT: calculator(input_expression="2+1")`
        - Output: `OBSERVE: 3`
        
* `get_activities_by_date_tool(date: str, city: str)`: Get the available activities and the cost given date and city
    - Example:
        - Input: `ACT: get_activities_by_date_tool(date="2025-06-10", city="AgentsVille")`
        - Output: `OBSERVE: [{{
                                "activity_id": "event-2025-06-10-1", 
                                "name": "Serve & Savor: Tennis and Taste Luncheon",
                                "start_time": "2025-06-10 12:00",
                                "end_time": "2025-06-10 13:30",
                                "location": "The Grand Racquet Terrace, AgentsVille",
                                "duration": "1.5 hours",
                                "price": 20,
                                "description": "Join us for 'Serve & Savor,' the ultimate crossover event for cooking and tennis enthusiasts in AgentsVille!....",
                             }}, 
                             {{
                                "activity_id": "event-2025-06-10-2",
                                "name": "Artful Athletics: Paint & Play Extravaganza",
                                "start_time": "2025-06-10 15:00",
                                "end_time": "2025-06-10 17:00",
                                "location": "Creative Courts Park, AgentsVille",
                                "price": 15,
                                "description": "Join us for an exciting afternoon at Creative Courts Park, where the worlds of art and sports collide!....",
                             }},..
                             ]`

* `run_evals_tool(vacation_info: VacationInfo, travel_plan: TravelPlan)`: Evaluate if the itinerary satisfies all the travel requirement, e.g. suitable to weather and budget etc. 
    - Example:
        - Input: `ACT: run_evals_tool(
                    vacation_info = {{"budget": 130,
                                     "date_of_arrival": "2025-06-10",
                                     "date_of_departure": "2025-06-12",
                                     "destination": "AgentsVille",
                                     "travelers": [{{"age": 30,
                                                     "interests": ["tennis", "cooking", "comedy", "technology"],
                                                     "name": "Yuri"}},  
                                    ]
                    }}
                    travel_plan={{
                                    "city":"AgentsVille", 
                                    "start_date":"2025-06-10",
                                    "end_date":"2025-06-12",
                                    "total_cost":60,
                                    "itinerary_days":[
                                        {{
                                          "date":"2025-06-10",
                                          "weather":{{"temperature":31.0,"temperature_unit":"celsius","condition":"clear"}},
                                          "activity_recommendations":[...]
                                        }}, ...
                                    ],
                                }}
                    )`
        - Output: `OBSERVE: {{"success": response.success, "failures": respone.failures}}`

* `final_answer_tool(final_output: TravelPlan)`: Return the final plan
    - Example:
        - Input: `ACT: final_answer_tool(
                    final_output={{
                                    "city":"AgentsVille", 
                                    "start_date":"2025-06-10",
                                    "end_date":"2025-06-12",
                                    "total_cost":60,
                                    "itinerary_days":[
                                        {{
                                          "date":"2025-06-10",
                                          "weather":{{"temperature":31.0,"temperature_unit":"celsius","condition":"clear"}},
                                          "activity_recommendations":[...]
                                        }}, ...
                                    ],
                        )`
        - Output: `OBSERVE: {{"city": "AgentsVille", "start_date":"2025-06-10", "end_date":"2025-06-12", "total_cost":60, ...}}`

You will not use any other tools.



## Output Format

Your response should include:

    * THOUGHT section outlines your reasoning and the steps you will take.
      ```json
      {{success=False, failure=[xxxx, yyyyyy, zzzzzzz]}}
      ```

    * ACTION section that specifies the tool calls in the following format:
      ```json
      {{"tool_name": "[tool_name]", "arguments": {{"arg1": "value1", ...}},....}}
      ```



## Context

- Weather Data:
{json.dumps(weather_for_dates, indent=2)}

- Activity Data:
{json.dumps(activities_for_dates, indent=2)}

- vacation info:
{vacation_info.model_dump_json(indent=2)}


## Example:

```
--USER MESSAGE--
TASK:
Respond to the query "Can you help me to plan a trip?. The start date is 2025-08-05 and end date is 2025-08-10.
and the city is "AgentsVille". My budget is $250."

--ASSISTANT MESSAGE--
THOUGHT:
    * We have a day-by-day plan. Check if the plan satisfies travelers' requirements and feedback.

ACTION:
    * To evaulate the plan: `run_evals_tool`:
      ```json
      {{
          "tool_name": ["run_evals_tool"], 
          "arguments": {{"vacation_info": vacation_info, "travel_plan": travel_plan}}
      }}
      ```

--USER MESSAGE--
OBSERVE: {{success=False, failures=["the third day only one activity was recommended, so the feedback was not completely incorporated throughout the plan."]}}

--ASSISTANT MESSAGE--
THOUGHT:
    * I see a failure since the third day has only one activity.
    * It is required at least two activities per day.
    * I need to call a tool to retrieve activity data for the day, and add an activity in the day.
    * After adding an activity, evaluate the plan again.

ACTION:
    * To get the activities for the day:
      ```json
      {{
          "tool_name": ["get_activities_by_date_tool"], 
          "arguments": {{"date: "2025-08-10",  "city": "AgentsVille"}}
      }}
      ```
    * Evaulate the plan again:
      {{
          "tool_name": ["run_evals_tool"], 
          "arguments": {{"vacation_info": vacation_info, "travel_plan": travel_plan}}
      }}


--USER MESSAGE--
OBSERVE: {{success=False, failures=["the estimated cost exceeds budget"]}}

--ASSISTANT MESSAGE--
THOUGHT:
    * I see a failure since the cost exceeds the budget.
    * We can replace an activitiy (for a certian day) to lower the cost.
    * I need to call a tool to retrieve activities for a day, and replace a cheaper activity in the plan.
    * After replacing an activity, evaluate the plan again.
    
ACTION:
    * To get the activities for a day:
      ```json
      {{
          "tool_name": ["get_activities_by_date_tool"], 
          "arguments": {{"date: "2025-08-11",  "city": "AgentsVille"}}
      }}
      ```
    * To evaulate the plan:
      {{
          "tool_name": ["run_evals_tool"], 
          "arguments": {{"vacation_info": vacation_info, "travel_plan": travel_plan}}
      }}

--USER MESSAGE--
OBSERVE: 
    {{success=True, failures=[]}}

--ASSISTANT MESSAGE--
THOUGHT:
    * The plan passed all eval. There is no failure.
    * It means the day-by-day plan satisfied to the travelers' requests and feedbacks.
    * I can return the final itinerary, call `final_answer_tool`, and then stop the ReAct: THINK → ACT → OBSERVE cycle.
    
ACTION:
    call `final_answer_tool`(final_output={{....}}), using 
    {{
          "tool_name": ["final_answer_tool"], 
          "arguments": {{"final_output": "travel_plan"}}
    }}


--USER MESSAGE--
OBSERVE: {{"city": "AgentsVille", "start_date":"2025-06-10", "end_date":"2025-06-12", "total_cost":60, itinerary_days: [...]...}}

```
"""



In [42]:
vacation_info

VacationInfo(travelers=[Traveler(name='Yuri', age=30, interests=[tennis, cooking, comedy, technology]), Traveler(name='Hiro', age=25, interests=[reading, music, theatre, art])], destination='AgentsVille', date_of_arrival=datetime.date(2025, 6, 10), date_of_departure=datetime.date(2025, 6, 12), budget=130)

In [34]:
from utils_travel_agent import ChatAgent


class ItineraryRevisionAgent(ChatAgent):
    system_prompt = ITINERARY_REVISION_AGENT_SYSTEM_PROMPT
    tools = ALL_TOOLS

    def get_observation_string(self, tool_call_obj) -> str:
        """Extracts the observation from the thought-action response."""

        if "tool_name" not in tool_call_obj:
            return "OBSERVATION: No tool name specified."

        if "arguments" not in tool_call_obj:
            return "OBSERVATION: No arguments specified."

        # If the arguments are not a dictionary, state the error
        if not isinstance(tool_call_obj["arguments"], dict):
            return f"OBSERVATION: Arguments should be a dictionary, got {type(tool_call_obj['arguments'])} instead."

        # If the tool name is not a string, state the error
        if not isinstance(tool_call_obj["tool_name"], str):
            return f"OBSERVATION: Tool name should be a string, got {type(tool_call_obj['tool_name'])} instead."

        tool_name = tool_call_obj["tool_name"]
        arguments = tool_call_obj["arguments"]

        tool_fn = None

        for tool in self.tools:
            if tool.__name__ == tool_name:
                tool_fn = tool
                break

        if tool_fn is None:
            return f"OBSERVATION: Unknown tool name '{tool_name}' in action string."

        try:
            tool_response = tool_fn(**arguments)
            return f"OBSERVATION: Tool {tool_name} called successfully with response: {tool_response}"
        except Exception as e:
            return f"OBSERVATION: Error occurred while calling tool {tool_name}: {e}"

    def run_react_cycle(
        self, original_travel_plan: TravelPlan, max_steps: int = 10, model: Optional[OpenAIModels] = None, client = None,
    ) -> TravelPlan:
        """Runs the ReAct cycle to revise the itinerary based on the evaluation results."""
        from json_repair import repair_json

        # Provide the original travel plan to revise
        self.add_message(
            role="user",
            content=f"Here is the itinerary for revision:\n{original_travel_plan.model_dump_json()}",
        )
        resp = None

        # Run the ReAct cycle for a maximum number of steps
        for step in range(max_steps):
            # Get the thought-action response from the agent
            resp = self.get_response(model=model, client=client) or ""

            # If there is no action, report it to the LLM and continue
            if "ACTION:" not in resp:
                self.add_message(role="user", content="No action found in response.")
                continue

            action_string = resp.split("ACTION:")[1].strip()

            # Parse the tool call JSON from the action string
            try:
                # Fix any JSON formatting issues. e.g. missing closing braces, etc.
                action_string = repair_json(action_string)
                tool_call_obj = json.loads(action_string)
            except json.JSONDecodeError:
                print(f"Invalid JSON in action string: {action_string}")
                self.add_message(
                    role="user",
                    content=f"Invalid JSON in action string: {action_string}",
                )
                continue

            tool_name = tool_call_obj.get("tool_name", None)

            # If the final answer tool is called, validate and return the final travel plan
            if tool_name == "final_answer_tool":
                try:
                    new_travel_plan = TravelPlan.model_validate(
                        tool_call_obj["arguments"].get("final_output", tool_call_obj["arguments"])
                    )
                    return new_travel_plan
                except Exception as e:
                    self.add_message(
                        role="user", content=f"Error validating final answer: {e}"
                    )
                    continue

            # For all other tools, execute the tool call and add the observation
            else:
                # Add the 
                observation_string = self.get_observation_string(
                    tool_call_obj=tool_call_obj
                )
                self.add_message(role="user", content=observation_string)

        raise RuntimeError(
            f"ReAct cycle did not complete within {max_steps} steps. Last response: {resp}"
        )


In [35]:
# Instantiate the Itinerary Revision Agent
itinerary_revision_agent = ItineraryRevisionAgent()

# Let's get a single THOUGHT/ACTION response back to check that the agent is working as expected.
resp = itinerary_revision_agent.chat(
    user_message=f"Here is the itinerary for revision: {travel_plan_1.model_dump_json(indent=2)}",
    add_to_messages=False,
    model=MODEL,
    client=client,
) or ""

print_in_box(resp, "Raw Response")
# Check for THOUGHT
if "THOUGHT:" in resp:
    print("✅ `THOUGHT:` found in raw the response, as expected.")
else:
    print("❌ Expected `THOUGHT:` in raw the response. Please check the system prompt (output format).")
# Check for ACTION
if "ACTION:" in resp:
    print("✅ `ACTION:` found in raw the response, as expected.")
else:
    print("❌ Expected `ACTION:` in raw the response. Please check the system prompt (output format).")
if "\"tool_name\"" in resp:
    print("✅ `\"tool_name\":` found in raw the response, as expected.")
else:
    print("❌ Expected `\"tool_name\":` in raw the response. Please check the system prompt (output format).")



╔══════════════════════════════════════[ ItineraryRevisionAgent - System Prompt ]══════════════════════════════════════╗
║ You are a travel planning expert and helping our customers to make a comprehensive travel plan iteratively using     ║
║ tool calls                                                                                                           ║
║ and reasoning in a multi-step process.                                                                               ║
║ ## Task Instructions:                                                                                                ║
║ - You will repeat the following step-by-step reasoning until the itinerary satisfies to travelers' requests and      ║
║ feedbacks:                                                                                                           ║
║     - THINKING: Look the day-by-day itinerary, analyze the problems, and determine what tools to call.               ║
║     - ACTING: Take a single t

In [36]:
# Now let's run the ReAct cycle multiple times to get the revised itinerary.
# Note: If this takes more than a few minutes, then the agent may be stuck in a loop.
# Examine the traces to understand where it is failing and see if adjusting the system prompt helps.
# Since LLMs are stochastic, you will get different results each time you run this cell.

itinerary_revision_agent = ItineraryRevisionAgent()
travel_plan_2 = itinerary_revision_agent.run_react_cycle(
    original_travel_plan=travel_plan_1, max_steps=20,
    model=MODEL,
    client=client,
)

print("✅ Revised itinerary generated successfully. Congratulations!")



╔══════════════════════════════════════[ ItineraryRevisionAgent - System Prompt ]══════════════════════════════════════╗
║ You are a travel planning expert and helping our customers to make a comprehensive travel plan iteratively using     ║
║ tool calls                                                                                                           ║
║ and reasoning in a multi-step process.                                                                               ║
║ ## Task Instructions:                                                                                                ║
║ - You will repeat the following step-by-step reasoning until the itinerary satisfies to travelers' requests and      ║
║ feedbacks:                                                                                                           ║
║     - THINKING: Look the day-by-day itinerary, analyze the problems, and determine what tools to call.               ║
║     - ACTING: Take a single t

In [37]:
travel_plan_2.total_cost

110

In [39]:
travel_plan_2.itinerary_days

[ItineraryDay(date=datetime.date(2025, 6, 10), weather=Weather(temperature=31.0, temperature_unit='celsius', condition='clear'), activity_recommendations=[ActivityRecommendation(activity=Activity(activity_id='event-2025-06-10-0', name='FutureTech Breakfast Meet-Up', start_time=datetime.datetime(2025, 6, 10, 9, 0), end_time=datetime.datetime(2025, 6, 10, 11, 0), location='The Innovation Atrium, Tech District, AgentsVille', description='Join fellow technology enthusiasts for a dynamic morning at the FutureTech Breakfast Meet-Up! Dive into the latest trends in tech, gadget demos, and networking opportunities over coffee and fresh pastries. Held indoors at the spacious Innovation Atrium, this event is perfect for tech lovers eager to exchange ideas and discover new possibilities in a comfortable, modern setting.', price=20, related_interests=[technology]), reasons_for_recommendation=["Matches Yuri's interest in technology", 'Indoor event suitable for warm weather', 'Good timing before lunc

In [38]:
# Last let's double check that the revised travel plan passes all evaluation functions.
# No changes needed here.

eval_results_2 = get_eval_results(
    vacation_info=vacation_info,
    final_output=travel_plan_2,
    eval_functions=ALL_EVAL_FUNCTIONS,
)

assert eval_results_2.success, f"❌ Read the traces above and modify the system prompt.\n\nFailures: {eval_results_2.failures}"

print("✅ All evaluation functions passed successfully for the revised travel plan.")

eval_results_2

✅ Traveler Yuri has a match with interest {technology} at FutureTech Breakfast Meet-Up
✅ Traveler Yuri has a match with interest {cooking, tennis} at Serve & Savor: Tennis and Taste Luncheon
✅ Traveler Yuri has a match with interest {cooking} at Palette & Palate: Art Meets Cooking Experience
✅ Traveler Yuri has a match with interest {technology} at Tech & Film Fusion Night
✅ Traveler Hiro has a match with interest {music} at Morning Groove Dance Party
✅ Traveler Hiro has a match with interest {art} at Palette & Palate: Art Meets Cooking Experience
✅ Traveler Hiro has a match with interest {music} at Soundtrack Picnic: Lunchtime Movies & Melodies
✅ Activity FutureTech Breakfast Meet-Up (on 2025-06-10) and weather 'clear' are compatible.
✅ Activity Serve & Savor: Tennis and Taste Luncheon (on 2025-06-10) and weather 'clear' are compatible.
✅ Activity Morning Groove Dance Party (on 2025-06-11) and weather 'partly cloudy' are compatible.
✅ Activity Palette & Palate: Art Meets Cooking Exper

EvaluationResults(success=True, failures=[], eval_functions=['eval_start_end_dates_match', 'eval_total_cost_is_accurate', 'eval_itinerary_events_match_actual_events', 'eval_itinerary_satisfies_interests', 'eval_total_cost_is_within_budget', 'eval_activities_and_weather_are_compatible', 'eval_traveler_feedback_is_incorporated'])

In [40]:
# Show the final travel plan in a readable format.

from IPython.display import display

for itinerary_day in travel_plan_2.itinerary_days:
    print(f"Date: {itinerary_day.date}")
    print(
        f"Weather: {itinerary_day.weather.condition} ({itinerary_day.weather.temperature}°{itinerary_day.weather.temperature_unit})"
    )

    activities_df = pd.DataFrame(
        [
            activity_recommendation.activity.model_dump()
            for activity_recommendation in itinerary_day.activity_recommendations
        ]
    )
    display(activities_df)

Date: 2025-06-10
Weather: clear (31.0°celsius)


,activity_id,name,start_time,end_time,location,description,price,related_interests
0,event-2025-06-10-0,FutureTech Breakfast Meet-Up,2025-06-10 09:00:00,2025-06-10 11:00:00,"The Innovation Atrium, Tech District, AgentsVille","Join fellow technology enthusiasts for a dynamic morning at the FutureTech Breakfast Meet-Up! Dive into the latest trends in tech, gadget demos, and networking opportunities over coffee and fresh pastries. Held indoors at the spacious Innovation Atrium, this event is perfect for tech lovers eager to exchange ideas and discover new possibilities in a comfortable, modern setting.",20,[technology]
1,event-2025-06-10-1,Serve & Savor: Tennis and Taste Luncheon,2025-06-10 12:00:00,2025-06-10 13:30:00,"The Grand Racquet Terrace, AgentsVille","Join us for 'Serve & Savor,' the ultimate crossover event for cooking and tennis enthusiasts in AgentsVille! Kick off your lunch hour with a friendly round of doubles on our outdoor courts, then unwind with a hands-on cooking workshop led by a local chef, where you'll prepare and enjoy delicious energy-boosting recipes. Whether you come for the sport or the flavors, this energizing luncheon celebrates both passions in a lively outdoor setting. Ideal for anyone who loves to play, cook, or simply savor fresh food and fun!",20,"[cooking, tennis]"


Date: 2025-06-11
Weather: partly cloudy (34.0°celsius)


,activity_id,name,start_time,end_time,location,description,price,related_interests
0,event-2025-06-11-0,Morning Groove Dance Party,2025-06-11 09:00:00,2025-06-11 10:30:00,"Rhythm Hall, Center Plaza, AgentsVille","Start your day with energy and joy at the Morning Groove Dance Party! This lively event welcomes dancers of all levels to join a vibrant indoor session filled with upbeat music and fun routines. Whether you love modern pop, Latin beats, or classic disco, our dance instructors will guide you to move and groove. Connect with fellow dance lovers in the colorful atmosphere of Rhythm Hall. Perfect for fans of dancing, music, and fitness. Let the rhythm move you! (Indoor event.)",15,"[dancing, music, fitness]"
1,event-2025-06-11-3,Palette & Palate: Art Meets Cooking Experience,2025-06-11 18:30:00,2025-06-11 20:30:00,"The Creative Canvas Studio, Artisanal Lane, AgentsVille","Immerse yourself in a colorful evening where art and cooking blend together! At 'Palette & Palate,' participants will begin indoors at The Creative Canvas Studio with a guided session to paint their own culinary-inspired masterpiece. Afterwards, a local chef will lead an interactive cooking class, teaching you how to craft vibrant, edible works of art. Whether you're an art enthusiast, a food lover, or both, this creative night is perfect for socializing and expressing yourself through color and flavor! All materials and ingredients are provided. This event is held indoors and welcomes all experience levels in art and cooking.",25,"[art, cooking]"


Date: 2025-06-12
Weather: thunderstorm (28.0°celsius)


,activity_id,name,start_time,end_time,location,description,price,related_interests
0,event-2025-06-12-1,Soundtrack Picnic: Lunchtime Movies & Melodies,2025-06-12 12:00:00,2025-06-12 13:30:00,"Starlight Amphitheater, AgentsVille","Experience the magic of classic movie scenes paired with live music at the outdoor Starlight Amphitheater! Bring your lunch and relax on the lawn as musicians perform iconic film soundtracks while selected clips light up our open-air screen. Perfect for movie buffs and music lovers alike, this engaging event celebrates both arts in a sunny lunchtime setting. In case of rain, the event will move indoors to the adjacent Harmony Hall. Come for the tunes, stay for the cinematic wonder!",15,"[movies, music]"
1,event-2025-06-12-3,Tech & Film Fusion Night,2025-06-12 19:00:00,2025-06-12 21:30:00,"Virtual Reality Theater, Silicon Plaza, AgentsVille","Dive into an immersive evening where the magic of movies meets the latest in technology! Join fellow movie buffs and tech enthusiasts for a special screening of cutting-edge sci-fi short films, followed by an interactive panel with local filmmakers and VR technologists. Experience the future of entertainment and discuss how technology is transforming the world of cinema. This exciting, indoor event at the Virtual Reality Theater is perfect for anyone interested in technology and movies.",15,"[technology, movies]"


In [44]:
def narrate_my_trip(vacation_info, itinerary, client, model, filename="/tmp/my_trip_narration.mp3"):
    from IPython.display import Audio, Markdown, display
    from openai import OpenAI

    resp = do_chat_completion(
        messages=[
            {
                "role": "user",
                "content": f"""
                Here is information on the trip collected by the Onboarding Agent:
                {vacation_info}.

                Here is the final itinerary:
                {itinerary}
                
                Introduce the trip (travelers, interests, restrictions, and total cost) and
                then discuss each day of the itinerary.

                Do not specify the cost of each activity.

                Do not reference the the narrative itself in the response.
                """,
            }
        ],
        client=client,
        model=model,
    )
    display(Markdown(resp))

    try:
        if resp:
            with client.audio.speech.with_streaming_response.create(
                model="gpt-4o-mini-tts",
                voice="coral",
                input=resp,
                instructions="Speak in a cheerful and positive tone.",
            ) as response:
                response.stream_to_file(filename)

            display(Audio(filename))
        else:
            print("No response from the chat completion API.")
    except Exception:
        pass


# And finally, just for fun, let's narrate the trip.

narrate_my_trip(
    vacation_info=vacation_info,
    itinerary=travel_plan_2,
    client=client,
    model=MODEL,  # Optionally, you can change the model here
)

The trip to AgentsVille features two travelers, Yuri and Hiro. Yuri is 30 years old with interests in tennis, cooking, comedy, and technology, while Hiro is 25 years old and enjoys reading, music, theatre, and art. The trip is planned for June 10 to June 12, 2025, and the total cost of the itinerary is well within their budget of 130.

On June 10, the weather will be clear with a warm temperature of 31°C. The day begins with the "FutureTech Breakfast Meet-Up," an indoor event at The Innovation Atrium ideal for technology enthusiasts like Yuri. It’s a great opportunity to explore the latest tech trends and network in a comfortable setting. Following that, the "Serve & Savor: Tennis and Taste Luncheon" takes place outdoors on the terrace, offering a unique blend of tennis and cooking activities which perfectly matches Yuri’s passions for both sports and culinary experiences. The outdoor setting suits the clear weather and provides a vibrant atmosphere to enjoy the day.

June 11 brings partly cloudy skies with a high of 34°C. The morning starts with the "Morning Groove Dance Party" at Rhythm Hall, an energetic indoor dance event focused on music and fitness. This activity complements Hiro’s love of music and adds a fun physical element for Yuri as well. In the evening, "Palette & Palate: Art Meets Cooking Experience" is scheduled at The Creative Canvas Studio. This creative workshop combines Hiro’s interest in art with Yuri’s passion for cooking, offering an interactive and social experience indoors, perfect for the partly cloudy conditions.

On June 12, the weather forecast predicts thunderstorms with a cooler temperature of 28°C. For midday, the "Soundtrack Picnic: Lunchtime Movies & Melodies" event at Starlight Amphitheater offers an engaging film and live music experience outdoors, with a rain plan to move indoors to Harmony Hall if needed. This event aligns well with both Hiro's love for music and movies. The evening features the "Tech & Film Fusion Night" at the Virtual Reality Theater, an immersive indoor event blending movies and technology. It provides a perfect conclusion to the trip, focusing on Yuri’s tech interests and Hiro’s passion for movies, and is well-suited to the stormy weather forecast.

Overall, the itinerary thoughtfully balances indoor and outdoor activities to match the expected weather, while carefully catering to both travelers’ varied interests for a rich and enjoyable experience in AgentsVille.